In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('/home/admin/Documents/98_model/src')
sys.path.append('/data01/Documents/98_model/src')

## Import

In [3]:
from project_paths import PROJECT_PATH, DATA_PATH, SRC_PATH
from target_columns.cols_small_y import (
    SMALL_Y_DEFECT_ELECTRODE,
    SMALL_Y_CTQ_ELECTRODE,
    SMALL_Y_DEFECT_WINDING,
    SMALL_Y_CTQ_WINDING,
    SMALL_Y_DEFECT_ASSEMBLY,
    SMALL_Y_CTQ_ASSEMBLY,
    SMALL_Y_DEFECT_WASHING,
    SMALL_Y_CTQ_ACTIVATION
)
import pandas as pd
from data_loader.data_loader import DataLoader
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score, mean_absolute_error
from tqdm import tqdm
import re
import warnings

warnings.filterwarnings('ignore')

## Load data by version

In [4]:
# idx = 0 
# data_loader = DataLoader()
# data = data_loader.load_data(version='v8')
# data.shape
# (59057, 25781)

In [5]:
from data_loader.data_loader import DataLoader
from features.feature_generator import FeatureGenerator
feature_generator = FeatureGenerator()

In [6]:
idx_len = 8

In [7]:
# idx = 0
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [8]:
# idx = 1
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [9]:
# idx = 2
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [10]:
# idx = 3
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [11]:
# idx = 4
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [12]:
# idx = 5
# data_loader = DataLoader()
# data = data_loader.load_data(version='v10-n32s-vm2')

# data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

# data = feature_generator.transform(data)
# data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
# del data

In [15]:
idx = 6
data_loader = DataLoader()
data = data_loader.load_data(version='v10-n32s-vm2')

data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

data = feature_generator.transform(data)
data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
del data

# DV02 : 주액 압력 / 와인더 텐션
# DV03 : 주액 압력 * 주액 시간 / 와인더 텐션
# DV06 : 주액 압력 *  워싱 온도 / 와인더 텐션
DV07 : 주액 압력 * 주액 시간 x 워싱 온도 / 와인더 텐션
# DV15 : 활성화 공정 종료 시간 - 조립 공정 종료 시간
# DV17 : 양극 음극 두께 잔차
# DV20 : 와인딩 속도 x 텐션 tradeoff
# DV27 : 누적 성형 카운트 square / log
# DV29 : 주해액 Lot 대비 Deviation
# DV30: SEI 형성량


0        [2026-03-20 06:51:07]
1        [2026-03-24 12:58:56]
2        [2026-03-24 12:58:56]
3        [2026-03-09 21:02:14]
4        [2026-03-25 10:17:47]
                 ...          
14824    [2026-03-19 14:19:24]
14825    [2026-03-20 06:51:07]
14826    [2026-03-20 06:51:07]
14827    [2026-03-18 07:28:19]
14828    [2026-03-18 07:28:19]
Name: 01_Mixing_Finished Date, Length: 14829, dtype: object

0       NaT
1       NaT
2       NaT
3       NaT
4       NaT
         ..
14824   NaT
14825   NaT
14826   NaT
14827   NaT
14828   NaT
Name: 07_Before Degas_Finished Date, Length: 14829, dtype: datetime64[ns]

# DV40: 활성화 종료 시간 - 믹싱 종료 시간
# DV42: 미세라인 (점도 * 고형분) + (코팅갭 * 슬러리 코팅온도)
# DV47: 전극주름 max텐션 - min텐션
# DV52: (양극 + 음극 + 분리막 두께) x 젤리롤 턴수
# DV56: (양극 + 음극 + 분리막 두께) x 젤리롤 턴수
# DV83: 양극 로딩량 * 음극 로딩량
# DV85: 와인더 감속 * 장력
# DV86: 와인더 가속 * 장력
# DV88: 


In [16]:
idx = 7
data_loader = DataLoader()
data = data_loader.load_data(version='v10-n32s-vm2')

data = data.loc[int(data.shape[0] * idx/idx_len):int(data.shape[0] * (idx+1)/idx_len), :]

data = feature_generator.transform(data)
data.to_parquet(f'feature_store_v10_n32s_{idx}.parquet')
del data

# DV02 : 주액 압력 / 와인더 텐션
# DV03 : 주액 압력 * 주액 시간 / 와인더 텐션
# DV06 : 주액 압력 *  워싱 온도 / 와인더 텐션
DV07 : 주액 압력 * 주액 시간 x 워싱 온도 / 와인더 텐션
# DV15 : 활성화 공정 종료 시간 - 조립 공정 종료 시간
# DV17 : 양극 음극 두께 잔차
# DV20 : 와인딩 속도 x 텐션 tradeoff
# DV27 : 누적 성형 카운트 square / log
# DV29 : 주해액 Lot 대비 Deviation
# DV30: SEI 형성량


0        [2026-03-18 07:28:19]
1        [2026-03-18 07:28:19]
2        [2026-03-20 06:51:07]
3        [2026-03-20 06:51:07]
4        [2026-03-20 06:51:07]
                 ...          
14824    [2026-03-20 15:59:56]
14825    [2026-03-24 15:31:54]
14826    [2026-03-24 12:58:56]
14827    [2026-03-24 12:58:56]
14828    [2026-03-24 15:31:54]
Name: 01_Mixing_Finished Date, Length: 14829, dtype: object

0       NaT
1       NaT
2       NaT
3       NaT
4       NaT
         ..
14824   NaT
14825   NaT
14826   NaT
14827   NaT
14828   NaT
Name: 07_Before Degas_Finished Date, Length: 14829, dtype: datetime64[ns]

# DV40: 활성화 종료 시간 - 믹싱 종료 시간
# DV42: 미세라인 (점도 * 고형분) + (코팅갭 * 슬러리 코팅온도)
# DV47: 전극주름 max텐션 - min텐션
# DV52: (양극 + 음극 + 분리막 두께) x 젤리롤 턴수
# DV56: (양극 + 음극 + 분리막 두께) x 젤리롤 턴수
# DV83: 양극 로딩량 * 음극 로딩량
# DV85: 와인더 감속 * 장력
# DV86: 와인더 가속 * 장력
# DV88: 


: 